In [5]:
import pandas as pd
from etl_sdt.utils.logging_config import logger 
def extract_single_sheet_excel(file_path, sheet_name):
    """
    Extracts and merges data from all sheets in an Excel file based on a key column.

    Args:
    - file_path (str): The path to the Excel file.
    - key_column (str): The column to use as the key for merging sheets.

    Returns:
    - pd.DataFrame: The merged data as a pandas DataFrame.
    """
    # try:
        # Read all sheets
    xls = pd.ExcelFile(file_path, engine='openpyxl')
    
    df = pd.read_excel(xls, sheet_name=sheet_name)
        
    # logger.info("Data extraction and merging completed successfully.")
    return df
    # except Exception as e:
    #     logger.error(f"Error occurred while extracting data from Excel file: {e}")
    #     return None

In [8]:
# from etl_sdt.extract.excel_extractor import *
# from etl_sdt.transform.data_transfomer import *
# from etl_sdt.config import *


df = extract_single_sheet_excel(file_path='/home/andrea/Desktop/Sarcoma-DT/ETL/data/250319_MDS.xlsx', sheet_name='MDS_Timo_KSW_LUKS')

/home/andrea/miniconda3/envs/sarcoma-dt/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [9]:
df

,Institution,Pat ID,Alt PID_Timo,Gender,Date of birth,General Consent agreed,Date of histological diagnosis,type_biopsy_Timo,biopsy_neoadjuvant_Timo,Grading (FNCLCC),...,Follow up SB presentation (Reason for SB presentation),(0) OP_619 - Reason for SB presentation_Timo,(0) OP_1113 - Specify treatment outside SwissSarcomaNetwork_Timo,(0) OP_1115 - Specify current status_Timo,(0) OP_1112 - Follow up SB presentation_Timo,(W) Other diagnoses?_Timo,"(newest) Patient history (clinics, therapy) - latest to newest_Timo",Date of last follow-up,Status,Bemerkungen
0,free field,number,free field,[1] male\n[2] female,YYYY,[0] no\n[1] yes\n[2] unknown,DD/MM/YYYY\n(=date_biopsy_Timo)\nAdjumed: Date...,0 no biopsy\n1 fine needle\n2 core biopsy (ima...,0 no\n1 yes,[0] Not a sarcoma\n[1] G1\n[2] G2\n[3] G3\n[4]...,...,[1] in context of 1st systemic recurrence (est...,1 first time presentation\n1 no prior interven...,1 after unplanned excision (whoops)\n2 partial...,1 initial diagnosis\n2 local recurrence\n3 sys...,1 in context of 1st systemic recurrence (estab...,free field,free field,DD/MM/YY\n=date_last_contact_Timo,[1] no evidence of disease (NED)\n[2] alive wi...,NaN
1,4,154911,NaN,male,1942,no,2021-01-27 00:00:00,NaN,NaN,G3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-06-22 00:00:00,[3] dead of disease (DOD),no falsch hinterlegt m.E. -> yes (Erstdiagnose...
2,4,154911,NaN,male,1942,NaN,27.01.2021,NaN,NaN,,...,3 in context of primary treatment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,156590,NaN,female,1959,NaN,18.02.2019,NaN,NaN,G3,...,,NaN,NaN,NaN,NaN,NaN,NaN,2020-10-28 00:00:00,3 dead of disease (DOD),NaN
4,4,156590,NaN,female,1959,NaN,18.02.2019,NaN,NaN,,...,3 in context of primary treatment,NaN,NaN,NaN,NaN,NaN,NaN,2020-10-28 00:00:00,3 dead of disease (DOD),NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2116,4,8476111,NaN,female,1988,yes,2023-11-14 00:00:00,NaN,NaN,[3] G3,...,NaN,NaN,NaN,4 no evidence of tumor,NaN,NaN,NaN,2025-01-31 00:00:00,[1] no evidence of disease (NED),NaN
2117,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2118,NaN,dunkelblau markierte Spalten gehören zum MDS u...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2119,NaN,hellblau markierte Spalten sind als Info aus d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
import pandas as pd

# Load the Excel file
file_path = '/home/andrea/Desktop/Sarcoma-DT/ETL/data/adjumed_export_20241027-120926.xlsx'
excel_data = pd.ExcelFile(file_path)

# Load each sheet
cases_df = excel_data.parse('Cases')
interventions_df = excel_data.parse('interventions')

# Ensure "Patient ID (PID)" is in each dataset; replace 'Patient ID (PID)' with the actual column name if different
pid_column = 'Patient ID (PID)'

# Sample 200 unique patients by Patient ID from each dataset
cases_sample = cases_df.drop_duplicates(subset=pid_column).sample(n=200, random_state=42)
interventions_sample = interventions_df.drop_duplicates(subset=pid_column).sample(n=200, random_state=42)

# Save the samples to a new Excel file for the test database
with pd.ExcelWriter('ETL/data/adjumed_export_20241027-120926_test.xlsx') as writer:
    cases_sample.to_excel(writer, sheet_name='Cases_Sample', index=False)
    interventions_sample.to_excel(writer, sheet_name='Interventions_Sample', index=False)

print("Sampled data saved to 'test_db_sample.xlsx'")


OSError: Cannot save file into a non-existent directory: 'ETL/data'

In [72]:
from langdetect import detect
from etl_sdt.transform.data_transfomer import DictionaryTransformer
from etl_sdt.transform.nlp_processor import PathologyNERExtractor
from etl_sdt.config import relevant_features
import time

text = ''
ner_genetic = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 

ner_medical = PathologyNERExtractor(model_name="Clinical-AI-Apollo/Medical-NER")   

In [ ]:

start = time.time()

for i in range(1):
    try:

        language = detect(text)


        DT = DictionaryTransformer(renaming_dict)
        translation = DT.detect_and_translate(text, 't5-small')


        ner_genetic.extract_entities(text)
        ner_medical.extract_entities(text) 
    except Exception as e:
        print(e)
stop = time.time()

print(f'Time elapsed: {stop - start}')



In [2]:
df.to_excel('C:\\Users\\Admin\\OneDrive - Hochschule Luzern\\Desktop\\Sarcoma-DT\\ETL\\data\\adjumed_export_test.xlsx', index=False)

In [ ]:
from etl_sdt.extract.excel_extractor import *
from etl_sdt.transform.data_transfomer import *
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *
from etl_sdt.config import *
from transformers import pipeline

df = pd.read_excel('data/adjumed_export_test.xlsx')

trs = DictionaryTransformer(renaming_dict)
df = trs.aggregate_dataframe(df)
df = trs.renaming(df)
df = trs.transform_dataframe(df, data_format_dict)

ner = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 
df = trs.report_info_extractor(df, ner)

ner_genetic = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 

ner_medical = PathologyNERExtractor(model_name="Clinical-AI-Apollo/Medical-NER") 

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-de-en")


free_text_columns = ['main_referral_diagnosis']

df = trs.free_text_info_extractor(df, free_text_columns, translator,  ner_genetic,  ner_medical)

records = trs.restructure_data(df, relevant_features, key_column= 'patient_id')

In [ ]:
records

In [ ]:
trs = DictionaryTransformer(renaming_dict)
df = trs.aggregate_dataframe(df)

In [ ]:
df

In [ ]:
df = trs.renaming(df)

In [ ]:
df

In [ ]:
df = trs.transform_dataframe(df, data_format_dict)

In [ ]:
df['report_radiation_oncology_upload']

In [ ]:
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *

ner = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 



df = trs.report_info_extractor(df, ner)

In [ ]:


ner_genetic = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 

ner_medical = PathologyNERExtractor(model_name="Clinical-AI-Apollo/Medical-NER") 
free_text_columns = ['main_referral_diagnosis']
df = trs.free_text_info_extractor(df, free_text_columns, ner_genetic,  ner_medical)

In [ ]:
records = trs.restructure_data(df, relevant_features, key_column= 'patient_id')

In [ ]:
records

In [ ]:
records[0]['doc_1']['operations']

In [ ]:
from pymongo import MongoClient
from datetime import datetime

# Establish a connection to MongoDB
client = MongoClient('mongodb://localhost:27017/')

# Select the database
db = client['SarcomaDB']
# Select the collection
collection = db['Patients']


def upsert_record(record):
    """
    Update the record if it exists, or insert it if it doesn't.
    
    Args:
    record (dict): The record to be inserted or updated.
    """
    try:
        # Define the query to find the document
        query = {"_id": record["_id"]}
        
        # Define the update operation
        update = {"$set": record}
        
        # Update the document if it exists, insert if it doesn't
        result = collection.update_one(query, update, upsert=True)
        
        if result.upserted_id:
            print(f"Inserted new document with _id: {result.upserted_id}")
        else:
            print(f"Updated existing document with _id: {record['_id']}")
    except Exception as e:
        print(f"Failed to upsert record with _id: {record['_id']}. Error: {e}")




for record in records: 
    upsert_record(record)


In [ ]:
from etl_sdt.extract.excel_extractor import *
from etl_sdt.transform.data_transfomer import *
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *
from etl_sdt.config import *
from transformers import pipeline

df = pd.read_excel('data/adjumed_export_test.xlsx')


trs = DictionaryTransformer(renaming_dict)
df = trs.aggregate_dataframe(df)
df = trs.renaming(df)
df = trs.transform_dataframe(df, data_format_dict)

ner = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 
df = trs.report_info_extractor(df, ner)

ner_genetic = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 

ner_medical = PathologyNERExtractor(model_name="Clinical-AI-Apollo/Medical-NER") 

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-de-en")


free_text_columns = ['summary_decision_further_strategy']

df = trs.free_text_info_extractor(df, free_text_columns, translator,  ner_genetic,  ner_medical)

In [ ]:
df['summary_decision_further_strategy_pdf']

In [25]:
import pandas as pd
from openpyxl import load_workbook

def append_to_excel(file_path, data):
    # Load existing workbook
    try:
        book = load_workbook(file_path)
        writer = pd.ExcelWriter(file_path, engine='openpyxl')
        writer.book = book
        
        # Read the existing data
        existing_df = pd.read_excel(file_path)
        
        # Create new data DataFrame
        new_data_df = pd.DataFrame(data)
        
        # Append new data to existing data
        updated_df = pd.concat([existing_df, new_data_df], ignore_index=True)
        
        # Write updated data back to Excel file
        updated_df.to_excel(writer, index=False)
        
        writer.save()
        writer.close()
        
        print(f"Data appended successfully to {file_path}")
    except FileNotFoundError:
        # If file doesn't exist, create it with new data
        new_data_df = pd.DataFrame(data)
        new_data_df.to_excel(file_path, index=False)
        print(f"Excel file created at {file_path} with new data")

In [ ]:
excel_path = "text_annotation_feedback.xlsx"
df_new = df[['summary_decision_further_strategy', 'summary_decision_further_strategy_pdf']]
column_mapping = {
    'main_referral_diagnosis': "Text Segment",
    'main_referral_diagnosis_pdf': "Annotations",
    'summary_files_upload': "Text Segment",
    'summary_files_upload_pdf': "Annotations",
    'summary_decision_further_strategy': "Text Segment",
    'summary_decision_further_strategy_pdf': "Annotations",
}
df_new.rename(columns=column_mapping, inplace=True)

df_new['Type'] = 'Main Referral Diagnosis'
df_new['Annotation'] = None
df_new['Feedback Comment'] = None

append_to_excel(excel_path, df)

In [ ]:
result = collection.insert_one(records[11])

In [ ]:
records[11]['doc_1']['diagnosis']

In [ ]:

# Define the query to find the document
query = {"_id": records[0]["_id"]}

# Define the update operation
update = {"$set": records[0]}
collection.update_one(query, update, upsert=True)

In [ ]:
def get_record_by_id(records, _id):
    for record in records:
        if record['_id'] == _id:
            return record
    return None

get_record_by_id(records, '9341344')


In [ ]:
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *


ner = PathologyNERExtractor(model_name="jnferfer/treatment-disease-NER") 

trs.ner_extract_treatment(df, ner)

In [ ]:
df

In [ ]:
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *

ner_genetic = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 

ner_medical = PathologyNERExtractor(model_name="Clinical-AI-Apollo/Medical-NER") 

df = trs.report_info_extractor(df, ner_genetic, ner_medical)

In [ ]:
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *

ner_genetic = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 

ner_medical = PathologyNERExtractor(model_name="Clinical-AI-Apollo/Medical-NER") 
free_text_columns = ['main_referral_diagnosis']
df = trs.free_text_info_extractor(df, free_text_columns, ner_genetic,  ner_medical)

In [ ]:
records[0]['doc_1'].keys()



In [ ]:
def check_duplicates_in_dict(dictionary):
    # Extract values from the dictionary
    values = list(dictionary.values())
    
    # Using a set to track unique values and a list to track duplicates
    seen = set()
    duplicates = set(value for value in values if value in seen or seen.add(value))
    
    # Check if duplicates list is not empty
    has_duplicates = len(duplicates) > 0
    
    # Return a boolean indicating if duplicates exist and the list of duplicates
    return has_duplicates, list(duplicates)
has_duplicates, duplicate_values = check_duplicates_in_dict(renaming_dict)
print("Duplicates exist:", has_duplicates)
print("Duplicate values:", duplicate_values)

In [ ]:
def restructure_data(df, relevant_features, key_column):

        records = []
        key_column = 'patient_id'
        for group_idx, (patient_id, group_df) in enumerate(df.groupby(key_column), start=1):
            
            patient_record = {"_id": patient_id}
            index_l=0
            for idx, (index, row) in enumerate(group_df.iterrows(), start=1):
                index_l+=1
                group_name=f'doc_{index_l}'
                patient_record[group_name] ={}
                
                for group, features in relevant_features.items():
                
                    group_data = []
                
                    
                    row_data = {}
                    for feature in features:
                        renamed_feature = trs.rename_dict.get(feature, feature)
                        try:
                            if row[renamed_feature] !=[] and row[renamed_feature] !=None :
                                if isinstance(row[renamed_feature], list):
                                    for item in row[renamed_feature]:
                                        if not pd.isna(item):
                                            row_data[renamed_feature] = row[renamed_feature]
                                else:
                                    if not pd.isna(row[renamed_feature]):
                                        row_data[renamed_feature] = row[renamed_feature]
                        except KeyError:
                            warnings.warn(f"Failed to allocate key: {renamed_feature}")
                    
                    
                    group_data.append(row_data)
                    
                    if '_n' in group:
                        group=group[:-2]
                        patient_record[group_name][group] = group_data
                    
                    else:
                        patient_record[group] = group_data
            records.append(patient_record)
        
        return records
trs.restructure_data(df_sample, relevant_features, key_column= 'patient_id')

In [ ]:
from etl_sdt.transform.nlp_processor import *
ner = PathologyNERExtractor(model_name="Clinical-AI-Apollo/Medical-NER") 

value = 'Comparatively to the investigation of 14.12. 2023, there is documented increase in size and metabolic stability of the subcutaneous nodular lesion at the level of the anterior right thigh showing central hypometabolic area secondary to involutional phenomena (current SUVmax 7). Stability of tumor disease interesting the quadriceps femoris muscle and proximal portions of the tibialis anterior muscle of the same limb.'

entities= ner.extract_entities(value)
# column_name = 'summary_decision_further_strategy'
# df, new_column_name = ner._rename_column_ner(df, column_name)
        

def extract_entities_dynamic(entities):
    pathology_info = {}
    
    current_entity = None
    current_text = []

    for entity in entities:
        entity_type = entity['entity']
        entity_text = entity['word'].replace('▁', ' ').strip()
        
        if entity_type.startswith('B-'):
            if current_entity is not None and current_text:
                category = current_entity.split('-')[-1].lower()
                if category not in pathology_info:
                    pathology_info[category] = []
                pathology_info[category].append("".join(current_text).replace('##', ''))

            current_entity = entity_type
            current_text = [entity_text]
        elif entity_type.startswith('I-') and current_entity is not None:
            current_text.append(entity_text)

    if current_entity is not None and current_text:
        category = current_entity.split('-')[-1].lower()
        if category not in pathology_info:
            pathology_info[category] = []
        pathology_info[category].append("".join(current_text).replace('##', ''))

    return pathology_info

annotations = extract_entities_dynamic(entities)
print(annotations)

In [ ]:
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *

pdf_path ='C:/Users/Admin/OneDrive - Hochschule Luzern/Desktop/Sarcoma-DT/ETL/data/pdfs/Histo definitiv.pdf'
value = extract_text_from_pdf(pdf_path, detect_vertical_text=False, word_margin=0.1)
ner = PathologyNERExtractor(model_name="Clinical-AI-Apollo/Medical-NER") 
entities= ner.extract_entities(value)
# column_name = 'summary_decision_further_strategy'
# df, new_column_name = ner._rename_column_ner(df, column_name)
        

def extract_entities_dynamic(entities):
    pathology_info = {}
    
    current_entity = None
    current_text = []

    for entity in entities:
        entity_type = entity['entity']
        entity_text = entity['word'].replace('▁', ' ').strip()
        
        if entity_type.startswith('B-'):
            if current_entity is not None and current_text:
                category = current_entity.split('-')[-1].lower()
                if category not in pathology_info:
                    pathology_info[category] = []
                pathology_info[category].append("".join(current_text).replace('##', ''))

            current_entity = entity_type
            current_text = [entity_text]
        elif entity_type.startswith('I-') and current_entity is not None:
            current_text.append(entity_text)

    if current_entity is not None and current_text:
        category = current_entity.split('-')[-1].lower()
        if category not in pathology_info:
            pathology_info[category] = []
        pathology_info[category].append("".join(current_text).replace('##', ''))

    return pathology_info

annotations = extract_entities_dynamic(entities)
print(annotations)

In [ ]:


annotations = extract_entities_dynamic(entities)
print(annotations)


In [ ]:
records = []
key_column = 'patient_id'
for group_idx, (patient_id, group_df) in enumerate(df.groupby(key_column), start=1):
    
    patient_record = {"_id": patient_id}
    index_l=0
    for idx, (index, row) in enumerate(group_df.iterrows(), start=1):
        index_l+=1
        group_name=f'doc_{index_l}'
        patient_record[group_name] ={}
        
        for group, features in relevant_features.items():
        
            group_data = []
        
            
            row_data = {}
            for feature in features:
                renamed_feature = trs.rename_dict.get(feature, feature)
                try:
                    if row[renamed_feature] !=[] and row[renamed_feature] !=None: row_data[renamed_feature] = row[renamed_feature]
                except KeyError:
                    warnings.warn(f"Failed to allocate key: {renamed_feature}")
            
            
            group_data.append(row_data)
            
            if '_n' in group:
                group=group[:-2]
                patient_record[group_name][group] = group_data
            
            else:
                patient_record[group] = group_data
    records.append(patient_record)

In [ ]:
trs.export_sample(records, 2)

In [ ]:
from etl_sdt.transform.data_transfomer import *
from etl_sdt.config import *

trs = DictionaryTransformer(rename_dict)

df = trs.aggregate_dataframe(df)
key_column = 'Patient ID (PID)'
records = trs.restructure_data(df, relevant_features, key_column)


# flat_features = trs._flatten_features(relevant_features)

# df_relevant = trs._filter_relevanat_fetaures( df, flat_features)
# df  = trs._rename_columns(df_relevant, rename_dict)

In [ ]:
import json
from etl_sdt.extract.excel_extractor import *
from etl_sdt.config import *
with open('buffer.json', 'r') as json_file:
    records = json.load(json_file)

In [ ]:
# process to transform in the right data format 

from etl_sdt.extract.excel_extractor import *
from etl_sdt.transform.data_transfomer import *
from etl_sdt.config import *

#1. import and aggregate data 
df = extract_excel(file_path='data/adjumed_export_20240607-174424.xlsx')
trs = DictionaryTransformer(rename_dict)
df = trs.aggregate_dataframe(df)

#2. rename the column
key_column = 'Patient ID (PID)'
df = trs.renaming(df, key_column)

df = trs.transform_dataframe(df, data_format_dict)
key_column = 'Patient ID (PID)'
records = trs.restructure_data(df, relevant_features, key_column)
trs.export_json(records, 'buffer.json')

In [ ]:
trs.export_sample(records, 2)

In [ ]:
# Define the conversion function
def convert_column(column, data_type):
    if data_type == 'datetime':
        return pd.to_datetime(column, errors='coerce', dayfirst=True)
    elif data_type == int:
        numeric_column = pd.to_numeric(column, errors='coerce')
        numeric_column = numeric_column.fillna(-1).astype(int).astype(pd.Int64Dtype())
        return numeric_column.replace(-1, None)
    elif data_type == float:
        return pd.to_numeric(column, errors='coerce')
    else:
        return column

# Define the main transformation function
def transform_dataframe(df, data_format_dict):
    # Convert columns to the appropriate data formats
    for column, data_type in data_format_dict.items():
        if column in df.columns:
            df[column] = convert_column(df[column], data_type)
    return df

# Example usage
data = {
    'age_at_admission': ['25', '30.0', '35', 'invalid', '40.5'],
    'admission_date': ['01-01-2020', '02-01-2020', 'invalid_date', '04-01-2020', '05-01-2020']
}
df = pd.DataFrame(data)
data_format_dict = {'age_at_admission': int, 'admission_date': 'datetime'}

df_new = transform_dataframe(df, data_format_dict)
print(df_new.age_at_admission)


In [ ]:
df_new['Date of biopsy (lowest of 806)_x']


In [ ]:
df_new['duration_of_session_minutes']


In [ ]:
# key_column = 'patient_id'
# df[key_column] = df[key_column].apply(lambda x: ','.join(map(str, x)) if isinstance(x, list) else str(x))
key_column = 'patient_id'
# df = trs._rename_columns(df, trs.rename_dict)

records = []
# key_column = 'patient_id'
for group_idx, (patient_id, group_df) in enumerate(df.groupby(key_column), start=1):
    
    patient_record = {"_id": patient_id}
    index_l=0
    for idx, (index, row) in enumerate(group_df.iterrows(), start=1):
        index_l+=1
        group_name=f'doc_{index_l}'
        patient_record[group_name] ={}
        
        for group, features in relevant_features.items():
        
            group_data = []
        
            
            row_data = {}
            for feature in features:
                renamed_feature = trs.rename_dict.get(feature, feature)
                try:

                    
                    if row[renamed_feature] !=[] and row[renamed_feature] !=None : row_data[renamed_feature] = row[renamed_feature]
                except KeyError:
                    warnings.warn(f"Failed to allocate key: {renamed_feature}")
            
            # Add an enumerated key to distinguish each row in the group
            group_data.append(row_data)
            
            patient_record[group_name][group] = group_data
    
    records.append(patient_record)

In [ ]:
records

In [ ]:
patient_counts = df.groupby('Patient ID (PID)').size().reset_index(name='counts')


patient_counts[patient_counts['counts'] > 2]


In [ ]:
key_column = 'patient_id'
aggregated_df = trs.aggregate_dataframe(df)

In [ ]:
flat_features = df.columns.tolist()
grouped_columns = trs.aggregate_columns_with_same_base(flat_features)
        
# Create a new DataFrame to store aggregated results
aggregated_data = {}

# Iterate through each group and sum the values row-wise
for base_name, group in grouped_columns.items():
    aggregated_data[base_name] = df[group].apply(lambda row: list(row.dropna()), axis=1)

# Create a new DataFrame with the aggregated data
aggregated_df = pd.DataFrame(aggregated_data)

df = aggregated_df.applymap(trs._extract_unique_values)

In [ ]:
records

In [ ]:
with open('buffer.json', 'r') as json_file:
    records = json.load(json_file)

In [ ]:
a =df.groupby(key_column)

a[1]

In [ ]:
key_column = 'patient_id'
df[key_column] = df[key_column].apply(lambda x: ','.join(map(str, x)) if isinstance(x, list) else str(x))

df = trs._rename_columns(df, rename_dict)

records = []
key_column = 'patient_id'
for group_idx, (patient_id, group_df) in enumerate(df.groupby(key_column), start=1):
    
    patient_record = {"_id": patient_id}
    index_l=0
    for idx, (index, row) in enumerate(group_df.iterrows(), start=1):
        index_l+=1
        group_name=f'doc_{index_l}'
        patient_record[group_name] ={}
        
        for group, features in relevant_features.items():
        
            group_data = []
        
            
            row_data = {}
            for feature in features:
                renamed_feature = trs.rename_dict.get(feature, feature)
                try:
                    if row[renamed_feature] !=[] and row[renamed_feature] !=None and pd.notnull(row[renamed_feature]) : row_data[renamed_feature] = row[renamed_feature]
                except KeyError:
                    warnings.warn(f"Failed to allocate key: {renamed_feature}")
            
            # Add an enumerated key to distinguish each row in the group
            group_data.append(row_data)
            
            patient_record[group_name][group] = group_data
    
    records.append(patient_record)

In [ ]:
import json
with open('buffer.json', 'w') as json_file:
    json.dump(records, json_file, indent=4)


In [ ]:
records = trs.restructure_data( aggregated_df, relevant_features, key_column)
trs.print_sample(records, 1)

In [ ]:
import itertools
df =trs._rename_columns(aggregated_df, rename_dict)
all_patient_ids = list(itertools.chain.from_iterable(df['patient_id']))

# Create a new DataFrame with the flattened patient_id values
flat_df = pd.DataFrame({'patient_id': all_patient_ids})

# Group by 'patient_id' and count the number of rows for each unique 'patient_id'
patient_row_counts = flat_df.groupby('patient_id').size().reset_index(name='row_count')

print(patient_row_counts)

In [ ]:
rows_with_two_or_more = df[df['patient_id'].apply(lambda x: len(x) >= 2)]

print(rows_with_two_or_more)

In [ ]:
trs.export_sample(records, 1)

In [ ]:
df = aggregated_df
df  = trs._rename_columns(df, trs.rename_dict)
records = []
for _, row in df.iterrows():
    patient_record = {}
    patient_id = row[key_column]
    for group, features in relevant_features.items():
        group_data = {}

        for feature in features:
            trs._key_allocation(group_data, row, feature)
        
        patient_record[group] = group_data
    
    patient_record["_id"] = patient_id
    records.append(patient_record)

In [ ]:
[item for sublist in relevant_features.values() for item in sublist]

In [ ]:
# restructure data
df = aggregated_df

# flat_features = trs._flatten_features( relevant_features)
# key_column = 'patient_id'
# df_relevant = trs._filter_relevanat_fetaures( df, flat_features)



In [ ]:
 
# df  = trs._rename_columns(df_relevant, rename_dict)

records = []
for _, row in df.iterrows():
    patient_record = {}
    patient_id = row[key_column]
    for group, features in relevant_features.items():
        group_data = {}

        for feature in features:
            renamed_feature = rename_dict.get(feature, feature)
            try:
                group_data[renamed_feature] = row[renamed_feature]
            except KeyError:
                warnings.warn(f"Failed to allocate key: {renamed_feature}")
        
        patient_record[group] = group_data
    
    patient_record["_id"] = patient_id
    records.append(patient_record)

In [ ]:
for i, (key) in enumerate(records):
    if i < 1:
        print(f"{key}")
    else:
        break

In [ ]:
n =1
with open('sample_docs.txt', 'w') as file:
# Print the first x documents and save them to the file
    for i, (key) in enumerate(records):
        if i < n:
            output = f"{key}\n"
            print(output.strip())
            file.write(output)
        else:
            break

In [ ]:
grouped_columns = trs.aggregate_columns_with_same_base(flat_features)
for base_name, group in grouped_columns.items():
    print(f"{base_name}: {group}")

In [ ]:
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *

pdf_path = "data/pdfs/histo definitiv.pdf"
pdf_text = extract_text_from_pdf(pdf_path, detect_vertical_text=False, word_margin=0.1)

ner = PathologyNERExtractor(model_name="alvaroalon2/biobert_genetic_ner") 
pdf_text ="""Striking and important in the present case is the anamnestic information that the tumor has existed since early childhood. 
since early childhood. Purely morphologically (“lipofibromatosis” aspect), the tumor is of the fibroblastic type. 
Fibroblastic form. Originally a so-called superficial CD34-
positive spindle cell tumor and a so-called “fibrous hamartoma of infancy” were originally considered. However 
However, the final immunophenotype (CD34+/S100+/panTRK+) suggests an NTRK-rearranged spindle cell tumor. 
Spindle cell tumor, a molecularly defined tumor entity that has been definitively included in the WHO classification of pediatric tumors. 
classification of pediatric tumors (chapter on soft tissue tumors), while it is included in the WHO 
classification of soft tissue tumors in the chapter on tumors with uncertain differentiation as “emerging”. 
Differentiation. In view of the long history of the disease and the extremely low 
proliferation rate, a benign tumor biology can be assumed. If desired, we can perform a 
molecular pathological confirmation of the NTRK fusion."""
entities = ner.extract_entities(pdf_text)

def extract_entities_dynamic(entities):
    pathology_info = {}
    
    current_entity = None
    current_text = []

    for entity in entities:
        entity_type = entity['entity']
        entity_text = entity['word'].replace('▁', ' ').strip()
        
        if entity_type.startswith('B-'):
            if current_entity is not None and current_text:
                category = current_entity.split('-')[-1].lower()
                if category not in pathology_info:
                    pathology_info[category] = []
                pathology_info[category].append("".join(current_text).replace('##', ''))

            current_entity = entity_type
            current_text = [entity_text]
        elif entity_type.startswith('I-') and current_entity is not None:
            current_text.append(entity_text)

    if current_entity is not None and current_text:
        category = current_entity.split('-')[-1].lower()
        if category not in pathology_info:
            pathology_info[category] = []
        pathology_info[category].append("".join(current_text).replace('##', ''))

    return pathology_info

annotations = extract_entities_dynamic(entities)
print(annotations)


In [8]:
# Use a pipeline as a high-level helper
from etl_sdt.extract.pdf_extractor import *
from etl_sdt.transform.nlp_processor import *

# pipe = pipeline("token-classification", model="pruas/BENT-PubMedBERT-NER-Gene")

ner = PathologyNERExtractor(model_name="Precious1/Clinical-Biomedical-Named-Entity-Recognition-Using-Scispacy") 

pdf_text ="""Striking and important in the present case is the anamnestic information that the tumor has existed since early childhood. 
since early childhood. Purely morphologically (“lipofibromatosis” aspect), the tumor is of the fibroblastic type. 
Fibroblastic form. Originally a so-called superficial CD34-
positive spindle cell tumor and a so-called “fibrous hamartoma of infancy” were originally considered. However 
However, the final immunophenotype (CD34+/S100+/panTRK+) suggests an NTRK-rearranged spindle cell tumor. 
Spindle cell tumor, a molecularly defined tumor entity that has been definitively included in the WHO classification of pediatric tumors. 
classification of pediatric tumors (chapter on soft tissue tumors), while it is included in the WHO 
classification of soft tissue tumors in the chapter on tumors with uncertain differentiation as “emerging”. 
Differentiation. In view of the long history of the disease and the extremely low 
proliferation rate, a benign tumor biology can be assumed. If desired, we can perform a 
molecular pathological confirmation of the NTRK fusion."""

entities = ner.extract_entities(pdf_text)

def extract_entities_dynamic(entities):
    pathology_info = {}
    
    current_entity = None
    current_text = []

    for entity in entities:
        entity_type = entity['entity']
        entity_text = entity['word'].replace('▁', ' ').strip()
        
        if entity_type.startswith('B'):
            if current_entity is not None and current_text:
                category = current_entity.split('-')[-1].lower()
                if category not in pathology_info:
                    pathology_info[category] = []
                pathology_info[category].append("".join(current_text).replace('##', ''))

            current_entity = entity_type
            current_text = [entity_text]
        elif entity_type.startswith('I') and current_entity is not None:
            current_text.append(entity_text)

    if current_entity is not None and current_text:
        category = current_entity.split('-')[-1].lower()
        if category not in pathology_info:
            pathology_info[category] = []
        pathology_info[category].append("".join(current_text).replace('##', ''))

    return pathology_info

def assemble_entities(entities):
    assembled_text = ""
    current_phrase = ""
    previous_end = None  # Track the end of the previous token for spacing

    for entity in entities:
        token_text = entity['word'].replace('##', '')  # Remove subword indicators
        token_start = entity['start']
        
        # Check if token is part of the same entity (continuous word or phrase)
        if previous_end is not None and token_start == previous_end:
            current_phrase += token_text
        else:
            # If a new word or phrase starts, add the previous phrase to the assembled text
            if current_phrase:
                assembled_text += f"{current_phrase} "
            current_phrase = token_text

        previous_end = entity['end']  # Update end position

    # Add the final phrase to the assembled text
    if current_phrase:
        assembled_text += current_phrase

    return assembled_text.strip() 

print(entities)
# annotations = extract_entities_dynamic(entities)
annotations = assemble_entities(entities)

print(annotations)

OSError: Precious1/Clinical-Biomedical-Named-Entity-Recognition-Using-Scispacy does not appear to have a file named config.json. Checkout 'https://huggingface.co/Precious1/Clinical-Biomedical-Named-Entity-Recognition-Using-Scispacy/tree/main' for available files.

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

# Load BioBERT model and tokenizer
model_name = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Initialize NER pipeline with BioBERT
nlp = pipeline("ner", model=model, tokenizer=tokenizer, grouped_entities=True)

# Sample genetic text to analyze
genetic_text = """Striking and important in the present case is the anamnestic information that the tumor has existed since early childhood. 
since early childhood. Purely morphologically (“lipofibromatosis” aspect), the tumor is of the fibroblastic type. 
Fibroblastic form. Originally a so-called superficial CD34-
positive spindle cell tumor and a so-called “fibrous hamartoma of infancy” were originally considered. However 
However, the final immunophenotype (CD34+/S100+/panTRK+) suggests an NTRK-rearranged spindle cell tumor. 
Spindle cell tumor, a molecularly defined tumor entity that has been definitively included in the WHO classification of pediatric tumors. 
classification of pediatric tumors (chapter on soft tissue tumors), while it is included in the WHO 
classification of soft tissue tumors in the chapter on tumors with uncertain differentiation as “emerging”. 
Differentiation. In view of the long history of the disease and the extremely low 
proliferation rate, a benign tumor biology can be assumed. If desired, we can perform a 
molecular pathological confirmation of the NTRK fusion."""

# Run NER on the text
entities = nlp(genetic_text)

# Display the recognized entities and their labels
print("Recognized Entities:")
for entity in entities:
    print(f"Entity: {entity['word']}, Label: {entity['entity_group']}, Score: {entity['score']:.2f}")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/andrea/miniconda3/envs/sarcoma-dt/lib/python3.12/site-packages/transformers/pipelines/token_classification.py:168: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Recognized Entities:
Entity: striking and important in, Label: LABEL_0, Score: 0.55
Entity: the, Label: LABEL_1, Score: 0.52
Entity: present, Label: LABEL_0, Score: 0.57
Entity: case is the, Label: LABEL_1, Score: 0.58
Entity: an, Label: LABEL_0, Score: 0.51
Entity: ##amnes, Label: LABEL_1, Score: 0.55
Entity: ##tic, Label: LABEL_0, Score: 0.52
Entity: information that the tumor has, Label: LABEL_1, Score: 0.57
Entity: existed, Label: LABEL_0, Score: 0.50
Entity: since, Label: LABEL_1, Score: 0.51
Entity: early, Label: LABEL_0, Score: 0.51
Entity: childhood, Label: LABEL_1, Score: 0.55
Entity: . since early, Label: LABEL_0, Score: 0.51
Entity: childhood., Label: LABEL_1, Score: 0.53
Entity: purely, Label: LABEL_0, Score: 0.54
Entity: morphological, Label: LABEL_1, Score: 0.53
Entity: ##ly, Label: LABEL_0, Score: 0.50
Entity: ( “, Label: LABEL_1, Score: 0.52
Entity: lip, Label: LABEL_0, Score: 0.51
Entity: ##ofibroma, Label: LABEL_1, Score: 0.55
Entity: ##tosis, Label: LABEL_0, Score: 0

In [ ]:
entities

In [ ]:
ner = PathologyNERExtractor(model_name="jnferfer/treatment-disease-NER") 



column_name = 'summary_decision_further_strategy'
df, new_column_name = ner._rename_column_ner(df, column_name)
        
for index, row in df.iterrows():
    value = row[column_name]
    annotations = []
    if value: 
        ner_tags= ner.extract_entities(value)
        tags =ner.extract_pathology_info( ner_tags)
        
    pathology_info = {
        "genetic": [],
    }

current_entity = None
current_text = []

for entity in entities:
    entity_type = entity['entity']
    entity_text = entity['word']

    if entity_type.startswith('B-t'):
        if current_entity is not None and current_text:
            category = current_entity.split('-')[-1].lower()
            pathology_info[category].append("".join(current_text).replace('##', ''))

        current_entity = entity_type
        current_text = [entity_text]
    elif entity_type.startswith('I-t') and current_entity is not None:
        current_text.append(entity_text)


if current_entity is not None and current_text:
    category = current_entity.split('-')[-1].lower()
    pathology_info[category].append("".join(current_text).replace('##', ''))
    
if  annotations:
    df.at[index, new_column_name] = [annotations]
else: 
    df.at[index, new_column_name] = None 

In [ ]:
from etl_sdt.transform.data_transfomer import *

transformer = DictionaryTransformer()
columns = ["column_1 (1)", "column_1 (2)", "column_2 (1)", "column_2"]
grouped_columns = transformer.aggregate_columns_with_same_base(columns)
for base_name, group in grouped_columns.items():
    print(f"{base_name}: {group}")